# Hyper Param Analysis

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import cartopy 

from load_tuning_results import (
    load_results_raw,
    load_result_for_key,
    add_derived_features,
    filter_suspicious_routes,
)

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
gdf = gpd.read_parquet("../results/results_prelim.geoparquet")
gdf = add_derived_features(gdf)
gdf

In [ ]:
gdf = filter_suspicious_routes(gdf)

In [ ]:
gdf.columns

## Influence of population size, crossover rounds, and mutation iterations

In [ ]:
def plot_elite_cost_hyper_params(
    gdf: gpd.GeoDataFrame = None,
    forcing_scenario_name: str = "baseline",
    secondary_legend: str = "hyper_crossover_rounds",
    relabs: str = "relative",
    hyper_hazard_penalty_multiplier: float = 0,
):
    _gdf = gdf.where(
        (gdf.hyper_hazard_penalty_multiplier == hyper_hazard_penalty_multiplier)
        & (gdf.forcing_scenario_name == forcing_scenario_name)
    ).dropna(how="all")
    _gdf["journey_cat"] = (
        _gdf["journey_name"].astype(str)
        + " | " + _gdf["journey_speed_knots"].astype(str) + " kn"
        + " | " + pd.to_datetime(_gdf["journey_time_start"]).dt.strftime("%Y-%m")
    ).astype('category')
    _q_gdf = (
        _gdf.groupby(["hyper_population_size", secondary_legend, "journey_name", "journey_speed_knots"])[f"elite_cost_{relabs}"]
        .quantile(np.linspace(0, 1, 31))
        .reset_index()
        .rename(columns={"level_4": "quantile", f"elite_cost_{relabs}": "value"})
    )
    sns.set_style("whitegrid")
    g = sns.relplot(
        data=_q_gdf,
        kind="line",
        x="quantile",
        y="value",
        hue="journey_speed_knots",
        row="journey_name",
        col="hyper_population_size",
        style=secondary_legend,
        marker="",
        height=3,
        aspect=1.5,
        linewidth=2,
        facet_kws={"sharey": True, "margin_titles": True},
    )
    g.set_axis_labels("Quantile", f"Elite cost ({relabs})")
    g.set_titles(col_template=f"{forcing_scenario_name}: population size: {{col_name}}", row_template="{row_name}")
    g.fig.savefig(f"../figures/03_hyper_cost_distribution_{secondary_legend}_{relabs}_scen-{forcing_scenario_name}.png", dpi=200)
    g.fig.savefig(f"../figures/03_hyper_cost_distribution_{secondary_legend}_{relabs}_scen-{forcing_scenario_name}.pdf", dpi=200)
    return g.fig

In [ ]:
for fsn in ["baseline"]:  # gdf.forcing_scenario_name.unique():
    for relabs in ["relative"]:  # ["absolute", "relative"]:
        for sl in ["hyper_generations", "hyper_crossover_rounds", "hyper_mutation_iterations"]:
            plot_elite_cost_hyper_params(gdf=gdf, relabs=relabs, forcing_scenario_name=fsn, secondary_legend=sl);